# 04_exploration — Questions métier, métadonnées et historique Delta
Requêtes SQL sur les tables `brz_` typées. Chaque question est suivie de sa requête puis d'une interprétation.


In [ ]:
%sql
USE CATALOG alpenia_dev;
USE SCHEMA bronze;
SET TIME ZONE 'UTC';

## Q1. Combien de contrats actifs par ville et par produit ?

In [ ]:
%sql
SELECT city,
       product,
       COUNT(*) AS active_contracts
FROM   brz_contracts
WHERE  status = 'active'
GROUP  BY city, product
ORDER  BY city, product;

Interprétation : sur les 4 000 contrats, 3 390 sont actifs (85 %). Dans les trois villes, l'auto domine l'habitation (environ 610 à 630 contrats auto contre 475 à 535 habitation) ; Luxembourg a le portefeuille actif le plus important (1 143 contrats), devant Paris (1 141) et Bruxelles (1 106).

## Q2. Nombre de sinistres, montant total réglé et montant moyen par sinistre, par ville
La table `brz_claims` ne contient pas la ville : on la récupère par jointure avec `brz_contracts` sur `contract_id` (ville de rattachement du contrat).

In [ ]:
%sql
SELECT ct.city,
       COUNT(*)                     AS claims_count,
       CAST(SUM(cl.amount_eur) AS DECIMAL(14,2)) AS total_settled_eur,
       CAST(AVG(cl.amount_eur) AS DECIMAL(10,2)) AS avg_amount_eur
FROM   brz_claims cl
JOIN   brz_contracts ct ON ct.contract_id = cl.contract_id
GROUP  BY ct.city
ORDER  BY total_settled_eur DESC;

In [ ]:
%sql
-- Contrôle : aucun sinistre perdu dans la jointure (contrat introuvable)
SELECT COUNT(*) AS claims_without_contract
FROM   brz_claims cl
LEFT ANTI JOIN brz_contracts ct ON ct.contract_id = cl.contract_id;

Interprétation : les 24 000 sinistres du trimestre représentent environ 43,2 M€ réglés. Luxembourg arrive en tête en nombre (8 062) comme en montant (14,63 M€, 1 814,62 € en moyenne) ; Bruxelles compte 7 892 sinistres pour 14,24 M€ (1 804,55 € en moyenne). Les trois villes sont très proches, aucune ne concentre la sinistralité. Le contrôle confirme qu'aucun sinistre n'est perdu dans la jointure.

## Q3. Délai moyen de règlement, en jours, par nature de sinistre

In [ ]:
%sql
SELECT peril,
       COUNT(*)                        AS claims_count,
       ROUND(AVG(settlement_days), 1)  AS avg_settlement_days,
       ROUND(MIN(settlement_days), 1)  AS min_days,
       ROUND(MAX(settlement_days), 1)  AS max_days
FROM   brz_claims
GROUP  BY peril
ORDER  BY avg_settlement_days DESC;

Interprétation : le délai moyen de règlement est d'environ 50 jours quelle que soit la nature du sinistre (de 49,8 j pour le bris de glace et le dégât des eaux à 51,1 j pour les catastrophes naturelles), avec des extrêmes de 0,8 à 100 jours partout. La nature du sinistre n'explique donc pas le délai sur ce jeu : il faudra chercher ailleurs (ville, montant, expertise).

## Q4. Les 10 clients ayant abandonné le plus de devis

In [ ]:
%sql
SELECT client_id,
       COUNT(*) AS abandoned_quotes
FROM   brz_web_events
WHERE  action = 'quote_abandoned'
GROUP  BY client_id
ORDER  BY abandoned_quotes DESC, client_id
LIMIT  10;

Interprétation : le client CL-001297 a abandonné 17 devis, suivi de CL-000207 (15) et de CL-000096 et CL-002019 (14 chacun). Ces clients, intéressés mais qui ne concluent pas, sont des cibles naturelles pour une relance commerciale ou une analyse du parcours de devis.

## Q5. Coût total des expertises par type de constat, et expertises encore ouvertes

In [ ]:
%sql
SELECT finding_type,
       COUNT(*)                            AS expertises_count,
       CAST(SUM(cost_eur) AS DECIMAL(12,2)) AS total_cost_eur,
       SUM(CASE WHEN is_open THEN 1 ELSE 0 END) AS open_expertises
FROM   brz_expertises
GROUP  BY finding_type
ORDER  BY total_cost_eur DESC;

In [ ]:
%sql
SELECT COUNT(*)                                 AS total_expertises,
       SUM(CASE WHEN is_open THEN 1 ELSE 0 END) AS open_expertises,
       CAST(SUM(cost_eur) AS DECIMAL(12,2))     AS total_cost_eur
FROM   brz_expertises;

Interprétation : les 450 expertises ont coûté 190 199,20 € au total. Les constats incendie (36 162,78 €) et catastrophe naturelle (35 233,16 €) sont les plus coûteux ; le dégât des eaux le moins (26 409,35 €). 52 expertises sont encore ouvertes (11,6 %), dont 12 en catastrophe naturelle et 10 en incendie : leur coût définitif n'est pas encore connu.

---
## Métadonnées et historique Delta — `brz_contracts`
Dans `DESCRIBE DETAIL` : `format` (= `delta`), `location` (emplacement de stockage géré par Unity Catalog), `numFiles`, `sizeInBytes`.
Dans `DESCRIBE HISTORY` : une ligne par version. Chaque exécution de `02_ingestion` produit une version `CREATE OR REPLACE TABLE AS SELECT` (ou `CREATE TABLE AS SELECT` pour la toute première), chaque exécution de `03_transformations` une autre version du même type (réécriture avec nouveau schéma).

In [ ]:
%sql
DESCRIBE DETAIL alpenia_dev.bronze.brz_contracts;

In [ ]:
%sql
DESCRIBE HISTORY alpenia_dev.bronze.brz_contracts;

In [ ]:
# Version lisible : quelle opération, quand, combien de lignes et de fichiers écrits
from pyspark.sql import functions as F
display(spark.sql("DESCRIBE HISTORY alpenia_dev.bronze.brz_contracts")
    .select("version", "timestamp", "operation",
            F.col("operationParameters")["mode"].alias("mode"),
            F.col("operationMetrics")["numOutputRows"].alias("rows_written"),
            F.col("operationMetrics")["numFiles"].alias("files_written"))
    .orderBy("version"))

Lecture de l'historique (exemple pour un enchaînement 02 → 02 → 03) :
| Version | Opération | Origine dans les notebooks |
|---|---|---|
| 0 | CREATE TABLE AS SELECT | 1ʳᵉ exécution de `02_ingestion` (création, colonnes STRING) |
| 1 | CREATE OR REPLACE TABLE AS SELECT | 2ᵉ exécution de `02_ingestion` (test d'idempotence, 4 000 lignes) |
| 2 | CREATE OR REPLACE TABLE AS SELECT | `03_transformations` (typage, nouveau schéma) |

---
## Bonus 1 — Time travel et RESTORE sur `brz_pricing`
Mise à jour volontairement erronée (primes de base de Paris doublées), lecture de la version précédente, puis restauration. Les numéros de version sont lus dynamiquement : la cellule reste ré-exécutable.

In [ ]:
T_PRICING = "alpenia_dev.bronze.brz_pricing"
v_before = spark.sql(f"DESCRIBE HISTORY {T_PRICING} LIMIT 1").first()["version"]
print("Version avant la mise à jour erronée :", v_before)

In [ ]:
%sql
-- Mise à jour erronée
UPDATE alpenia_dev.bronze.brz_pricing
SET    base_premium_eur = base_premium_eur * 2
WHERE  city = 'Paris';

In [ ]:
print("Version actuelle (erronée)")
display(spark.table(T_PRICING).orderBy("city", "product"))

print(f"Version {v_before} (avant la mise à jour)")
display(spark.sql(f"SELECT * FROM {T_PRICING} VERSION AS OF {v_before} ORDER BY city, product"))

In [ ]:
spark.sql(f"RESTORE TABLE {T_PRICING} TO VERSION AS OF {v_before}")
display(spark.table(T_PRICING).orderBy("city", "product"))

In [ ]:
display(spark.sql(f"DESCRIBE HISTORY {T_PRICING}")
    .select("version", "timestamp", "operation", "operationParameters")
    .orderBy("version"))

## Bonus 2 — OPTIMIZE sur `brz_web_events`
`OPTIMIZE` compacte les petits fichiers en fichiers plus gros : moins de fichiers à ouvrir à la lecture, donc des requêtes plus rapides.

In [ ]:
T_WEB = "alpenia_dev.bronze.brz_web_events"
files_before = spark.sql(f"DESCRIBE DETAIL {T_WEB}").first()["numFiles"]
display(spark.sql(f"OPTIMIZE {T_WEB}"))
files_after = spark.sql(f"DESCRIBE DETAIL {T_WEB}").first()["numFiles"]
print(f"numFiles avant OPTIMIZE : {files_before}  ->  après : {files_after}")

Explication : `numFiles` de `DESCRIBE DETAIL` compte les fichiers de la version courante. Si la table a été écrite en plusieurs petits fichiers, OPTIMIZE les fusionne : le nombre diminue. Les anciens fichiers ne sont pas supprimés du stockage (toujours utiles au time travel) : ils sont seulement retirés de la version courante. Si la table ne contenait déjà qu'un seul fichier, OPTIMIZE ne fait rien (sur serverless, l'écriture optimisée et l'auto-compaction produisent souvent déjà des fichiers bien dimensionnés).

## Bonus 3 — VACUUM sur `brz_pricing` puis time travel

In [ ]:
%sql
VACUUM alpenia_dev.bronze.brz_pricing;

In [ ]:
# Time travel vers la version d'avant la mise à jour erronée, après VACUUM
display(spark.sql(f"SELECT * FROM {T_PRICING} VERSION AS OF {v_before} ORDER BY city, product"))

Observation : le time travel fonctionne encore. `VACUUM` ne supprime que les fichiers qui ne sont plus référencés par la version courante et plus anciens que la durée de rétention (7 jours par défaut, `delta.deletedFileRetentionDuration`). Les fichiers de la version d'avant l'UPDATE ont quelques minutes : ils sont conservés.
Le time travel vers cette version cesserait de fonctionner dès qu'un `VACUUM` serait lancé plus de 7 jours après que ses fichiers ont été remplacés (ou avec une rétention réduite, ex. `RETAIN 0 HOURS`, ce que Databricks bloque par défaut par sécurité). L'historique (`DESCRIBE HISTORY`) resterait visible, mais la lecture échouerait faute de fichiers.